In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Step 1: Import Libraries**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

print(tf.__version__)

**Step 2: Check GPU**

In [ ]:
print("GPU Available:")
print(tf.config.list_physical_devices('GPU'))

**DATASET DOWNLOARD**

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy"
)

print(path)

**Step 3: Define Dataset Paths**

In [ ]:
train_dir = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/train"

val_dir = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/val"

test_dir = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/dr_unified_v2/dr_unified_v2/test"

**Step 4: Check Dataset Structure**

In [ ]:
for folder in [train_dir, val_dir, test_dir]:

    print("\n", folder)

    total = 0

    for c in sorted(os.listdir(folder)):

        n = len(os.listdir(os.path.join(folder,c)))

        print(f"Class {c}: {n}")

        total += n

    print("Total:", total)

**Step 5: Image preprocessing and augmentation**

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 5

Training generator:

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

Validation/Test:

In [ ]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

**Step 6: Create Generators**

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = test_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

**Step 7: Display Class Labels**

In [ ]:
print(train_generator.class_indices)

**Step 8: Build MobileNetV2 Model**

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

Freeze layers:

In [ ]:
base_model.trainable = False

**Add classification layers:**

In [ ]:
x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(512,activation='relu')(x)

x = Dropout(0.5)(x)

output = Dense(
    NUM_CLASSES,
    activation='softmax'
)(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

**Step 9: Compile Model**

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

**Step 11: Callbacks**

In [ ]:
checkpoint = ModelCheckpoint(
    "mobilenetv2_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    verbose=1
)

**Step 12: Train Model**

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

Evaluate on Test Set

In [ ]:
results = model.evaluate(test_generator)

print("Test Accuracy:", results[1]*100)
print("Test Loss:", results[0])

Generate Accuracy and Loss Graphs

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.legend(['Train','Validation'])
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.legend(['Train','Validation'])
plt.show()

In [ ]:
plt.savefig("accuracy_graph.png")
plt.savefig("loss_graph.png")

Generate Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

pred = model.predict(test_generator)
pred_classes = np.argmax(pred, axis=1)

cm = confusion_matrix(
    test_generator.classes,
    pred_classes
)

sns.heatmap(cm, annot=True, fmt='d')
plt.savefig("confusion_matrix.png")

In [ ]:
model.save("/kaggle/working/mobilenetv2_best_77.keras")

In [ ]:
import os

print(os.path.exists(
    "/kaggle/working/mobilenetv2_best_77.keras"
))

In [ ]:
!ls -lh /kaggle/working

In [ ]:
from IPython.display import FileLink

FileLink(
    "/kaggle/working/mobilenetv2_best_77.keras"
)